# 🏔️ PeakInfer Demo - Analyze TinyLLM on Colab

This notebook demonstrates PeakInfer analyzing a vLLM-based project (TinyLLM) on Google Colab with GPU.

In [ ]:
# Step 1: Install Node.js 20
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs
!node --version
!npm --version

In [ ]:
# Step 2: Clone and Build PeakInfer from source
!git clone https://github.com/kalmantic/peakinfer.git
%cd peakinfer
!npm install
!npm run build
!npm link  # Makes 'peakinfer' available globally

In [ ]:
# Step 3: Verify PeakInfer installation
!peakinfer --help

In [ ]:
# Step 4: Set Anthropic API Key
import os
from google.colab import userdata

# Option A: Set in Colab Secrets (key icon in left sidebar) - RECOMMENDED
try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('✅ API Key loaded from Colab Secrets!')
except:
    print('⚠️ Add ANTHROPIC_API_KEY to Colab Secrets (key icon in left sidebar)')
    print('   Or set manually below:')

# Option B: Set manually (uncomment and add your key)
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-api03-YOUR-KEY-HERE'

In [ ]:
# Step 5: Clone TinyLLM (sample vLLM project to analyze)
%cd /content
!git clone https://github.com/jasonacox/TinyLLM.git
!ls TinyLLM/

In [ ]:
# Step 6: Check GPU availability
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# Step 7: Run PeakInfer Analysis on TinyLLM
# Analyze the chatbot app (contains the main LLM code)
!peakinfer analyze /content/TinyLLM/chatbot/app

In [ ]:
# Step 8: Run PeakInfer Recommend for cost optimization
!peakinfer recommend /content/TinyLLM/chatbot/app

In [ ]:
# Step 9: View the generated reports
import json

# StackMap
try:
    with open('/content/TinyLLM/chatbot/app/peakinfer-stackmap.json') as f:
        stackmap = json.load(f)
        print('📊 StackMap:')
        print(json.dumps(stackmap, indent=2)[:2000])
except FileNotFoundError:
    print('No stackmap generated yet')

print('\n' + '='*50 + '\n')

# Recommendations
try:
    with open('/content/TinyLLM/chatbot/app/peakinfer-recommendations.json') as f:
        recs = json.load(f)
        print('💡 Recommendations:')
        print(f"Total Callsites: {recs.get('totalCallsites', 0)}")
        print(f"Current Monthly Cost: ${recs.get('totalCurrentMonthlyCost', 0):.2f}")
        print(f"Recommended Monthly Cost: ${recs.get('totalRecommendedMonthlyCost', 0):.2f}")
        print(f"Potential Savings: ${recs.get('totalMonthlySavings', 0):.2f}/mo ({recs.get('savingsPercent', 0):.0f}%)")
except FileNotFoundError:
    print('No recommendations generated yet')

## Optional: Install and Run vLLM

If you want to actually run vLLM inference on Colab:

In [ ]:
# Optional: Install vLLM (takes ~5 minutes)
# !pip install vllm

# Start vLLM server with a small model
# !python -m vllm.entrypoints.openai.api_server \
#     --model facebook/opt-125m \
#     --port 8000 \
#     --gpu-memory-utilization 0.5